## Benchmark: Pitch Detection

This is the 1st of 3 component benchmarking we'll be conducting to test the general performance of the TuneBuddy system.

1. Pitch detection
2. Note detection
3. App usefulness

In [1]:
# autoreload when modules edited
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('..')

# from app_logic.user.AudioRecorder import AudioRecorder
# from app_logic.user.AudioPlayer import AudioPlayer

from algorithms.PitchDetector import PitchDetector
from algorithms.NoteDetector import NoteDetector
from algorithms.StringEditor import StringEditor
from algorithms.Config import Config
from app_logic.user.ds.Recording import Recording

Notes on the MIREX evaluation system:
- Accuracy based on whether the estimated pitch is within 50 cents (half a semitone) of the true pitch.


The metrics:
- Voicing Recall
- Voicing False Alarm
- Raw Pitch Accuracy
- Raw Chroma Accuracy
- Overall Accuracy

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import mir_eval

class Benchmarker:
    def __init__(self, dataset: str = "mdb-stem-synth"):
        self.dataset = dataset
        self.ANNOT_DIR = Path(f"../resources/benchmark/{dataset}/annot/")
        self.AUDIO_DIR = Path(f"../resources/benchmark/{dataset}/audio/")
        self.tracks = list(self._get_track_iter())

        config = {
            'sr': 44100,    # sample rate
            'w1': 1024 * 8,  # frame size
            'h1': 128,       # hop size
            'fmin': 20.0,
            'fmax': 1500.0,
            'tuning': 440.0,
            'unv_thresh': 0.8, # if unvoiced_prob > unv_thresh, consider the frame unvoiced

            # --- NOTE DETECTION PARAMETERS ---
            'w2': 30, # frame size
            'h2': 27, # hop size
            'pitch_thresh': 0.75,
            'slope_thresh': 1.5,
            'unv_ratio': 0.5, # proportion of unvoiced pitches in a window to consider the window unvoiced

            # --- STRING EDIT PARAMETERS ---
            'ins_cost': 1.5,
            'del_cost': 2,
            'sub_cost': 1,
            'tolerance': 1,
            # tiger-mom parameter
            'tiger_level': 1
        }
        self.config = Config(**config)
        self.pd, self.nd, self.se = (
            PitchDetector(config=self.config), 
            NoteDetector(config=self.config), 
            StringEditor(config=self.config)
        )

    def parse_annot_csv(self, annot_path: Path) -> tuple[np.ndarray, np.ndarray]:
        """
        Parse annotation CSV file into mir_eval friendly format.
        """
        df = pd.read_csv(
            annot_path,
            sep=r"\s+|,",
            header=None,
            engine='python'
        )
        times = df[0].values
        freqs = df[1].values
        return times, freqs
    
    def parse_pitch_data(self, recording: Recording) -> tuple[np.ndarray, np.ndarray]:
        """
        Parse Recording into mir_eval friendly format.
        """
        times = []
        freqs = []
        for p in recording.pitch_data.data:
            times.append(p.time)
            freq = self.config.midi_to_freq(p.candidates[0][0]) if p.unvoiced_prob < self.config.unv_thresh else 0.0
            freqs.append(freq)
        return np.array(times), np.array(freqs)

    def _get_track_iter(self):
        """
        Yield (track_id, audio_path, annot_path) for all audio files
        that have a matching annotation CSV.
        """
        for audio_path in sorted(self.AUDIO_DIR.glob("*.wav")):
            track_id = audio_path.stem  # e.g. "AClassicEducation_NightOwl_STEM_01.RESYN"
            annot_path = self.ANNOT_DIR / f"{track_id}.csv"
            if annot_path.exists():
                yield track_id, audio_path, annot_path

    def benchmark_pitch_detection(self, max_tracks: int=None, verbose: bool=True, save_path: str=None) -> pd.DataFrame:
        print(f"Benchmarking Pitch Detection on Dataset: {self.dataset}...")
        results = []

        max_tracks = max_tracks if max_tracks is not None else len(self.tracks)

        for i, (track_id, audio_path, annot_path) in enumerate(self.tracks):
            if i >= max_tracks:
                break
            print(f"Track {i+1}/{max_tracks}: {track_id}")
            recording = Recording()
            recording.load_audio(audio_path)
            ref_times, ref_freqs = self.parse_annot_csv(annot_path)
            est_times, est_freqs = self.parse_pitch_data(recording)
            
            metrics = mir_eval.melody.evaluate(ref_times, ref_freqs, est_times, est_freqs)
            
            if verbose: # print all the metrics we computed
                for k, v in metrics.items():
                    print(f"     {k}: {v:.4f}")
            
            row = {"track_id": track_id, **metrics}
            results.append(row)

        df = pd.DataFrame(results).set_index("track_id")

        # save to CSV if supplied
        if save_path is not None:
            df.to_csv(save_path, index=True)
            print(f"Results saved to {save_path}")

        return df

    
    def print_tracks(self):
        for track_id, audio_path, annot_path in self.tracks:
            print(f"Track ID: {track_id}")
            print(f"Audio Path: {audio_path}")
            print(f"Annotation Path: {annot_path}")
            print("-----")

### Benchmark pitch detection across datasets:

Rk: All datasets have monophonic music signals. We use the `mir_eval` framework to evaluate, eg the standard "music information retrieval" evaluation metrics.

We use the following datasets, from Salamon et. al from NYU + UPF.

- bach10-mf10-synth
- mdb-melody-synth
- mdb-mf0-synth
- mdb-stem-synth

In [6]:
benchmarker = Benchmarker(dataset="mdb-melody-synth")
df = benchmarker.benchmark_pitch_detection(max_tracks=3, verbose=True)

Benchmarking Pitch Detection on Dataset: mdb-melody-synth...
Track 1/3: AClassicEducation_NightOwl_STEM_08.RESYN


100%|██████████| 59051/59051 [00:17<00:00, 3321.48it/s] 


     Voicing Recall: 0.7210
     Voicing False Alarm: 0.0073
     Raw Pitch Accuracy: 0.6623
     Raw Chroma Accuracy: 0.6782
     Overall Accuracy: 0.8329
Track 2/3: AimeeNorwich_Child_STEM_04.RESYN


100%|██████████| 65194/65194 [00:15<00:00, 4107.40it/s] 


     Voicing Recall: 0.9681
     Voicing False Alarm: 0.0262
     Raw Pitch Accuracy: 0.9336
     Raw Chroma Accuracy: 0.9338
     Overall Accuracy: 0.9574
Track 3/3: AlexanderRoss_GoodbyeBolero_STEM_06.RESYN


100%|██████████| 144280/144280 [00:30<00:00, 4691.95it/s] 


     Voicing Recall: 0.3723
     Voicing False Alarm: 0.0032
     Raw Pitch Accuracy: 0.3131
     Raw Chroma Accuracy: 0.3146
     Overall Accuracy: 0.7780


In [ ]:
# run on the old thing
benchmarker = Benchmarker(dataset="mdb-melody-synth")
df = benchmarker.benchmark_pitch_detection(max_tracks=3, verbose=True)

Benchmarking Pitch Detection on Dataset: mdb-melody-synth...
Track 1/3: AClassicEducation_NightOwl_STEM_08.RESYN


100%|██████████| 59051/59051 [00:17<00:00, 3424.39it/s] 


     Voicing Recall: 0.7454
     Voicing False Alarm: 0.0098
     Raw Pitch Accuracy: 0.6582
     Raw Chroma Accuracy: 0.6850
     Overall Accuracy: 0.8297
Track 2/3: AimeeNorwich_Child_STEM_04.RESYN


100%|██████████| 65194/65194 [00:15<00:00, 4127.35it/s] 


     Voicing Recall: 0.9741
     Voicing False Alarm: 0.0279
     Raw Pitch Accuracy: 0.9365
     Raw Chroma Accuracy: 0.9369
     Overall Accuracy: 0.9576
Track 3/3: AlexanderRoss_GoodbyeBolero_STEM_06.RESYN


100%|██████████| 144280/144280 [00:30<00:00, 4794.39it/s] 


     Voicing Recall: 0.4910
     Voicing False Alarm: 0.0070
     Raw Pitch Accuracy: 0.3166
     Raw Chroma Accuracy: 0.3250
     Overall Accuracy: 0.7765


In [25]:
benchmarker = Benchmarker(dataset="bach10-mf0-synth")
df1 = benchmarker.benchmark_pitch_detection(max_tracks=3, verbose=True)

Benchmarking Pitch Detection on Dataset: bach10-mf0-synth...
Track 1/3: 01_AchGottundHerr_bassoon.RESYN


100%|██████████| 8628/8628 [00:11<00:00, 754.57it/s] 


     Voicing Recall: 0.9275
     Voicing False Alarm: 0.3186
     Raw Pitch Accuracy: 0.7693
     Raw Chroma Accuracy: 0.8202
     Overall Accuracy: 0.7629
Track 2/3: 01_AchGottundHerr_clarinet.RESYN


100%|██████████| 8628/8628 [00:10<00:00, 834.21it/s] 


     Voicing Recall: 0.9967
     Voicing False Alarm: 0.2628
     Raw Pitch Accuracy: 0.8908
     Raw Chroma Accuracy: 0.8926
     Overall Accuracy: 0.8749
Track 3/3: 01_AchGottundHerr_saxphone.RESYN


100%|██████████| 8628/8628 [00:10<00:00, 824.27it/s] 

     Voicing Recall: 0.9808
     Voicing False Alarm: 0.2558
     Raw Pitch Accuracy: 0.8432
     Raw Chroma Accuracy: 0.8433
     Overall Accuracy: 0.8354
